In [1]:
import numpy as np
from river import tree, metrics
from river.stream import iter_array
from SDDM import FullSDDM

def run_sddm_test():
    # 1. Generowanie syntetycznego strumienia danych z dryfem
    np.random.seed(42)
    n_samples = 4000
    batch_size = 200
    
    # 3 cechy (wartości od 0 do 1)
    X = np.random.rand(n_samples, 3) 
    y = np.zeros(n_samples, dtype=int)
    
    drift_point = 2000
    feature_names = ["cecha_0", "cecha_1", "cecha_2"]
    
    # Tworzenie koncepcji
    for i in range(n_samples):
        if i < drift_point:
            # Przed dryfem: wynik zależy od "cecha_0"
            y[i] = 1 if X[i, 0] > 0.5 else 0
        else:
            # Po dryfie: wynik zależy od "cecha_1" (real concept drift)
            y[i] = 1 if X[i, 1] > 0.5 else 0
            
    # Konwersja numpy do strumienia w formacie słowników, jakiego używa River
    dataset = iter_array(X, y, feature_names=feature_names)

    # 2. Inicjalizacja środowiska testowego
    model = tree.HoeffdingTreeClassifier()
    metric = metrics.Accuracy()
    
    sddm = FullSDDM(
        p_size=batch_size,
        num_features=3,
        num_classes=2,
        drift_type="abrupt"  # próg 0.6
    )
    
    # Bufory potrzebne dla SDDM (wymaga batchy, River działa instancja po instancji)
    X_batch = []
    y_batch = []
    batch_count = 0
    
    print("Rozpoczęcie strumieniowania danych...\n")
    
    # 3. Pętla strumieniowa (Prequential Evaluation)
    for i, (x_dict, y_true) in enumerate(dataset):
        
        # --- ETAP A: Działanie klasyfikatora ---
        # Test
        y_pred = model.predict_one(x_dict)
        if y_pred is not None:
            metric.update(y_true, y_pred)
            
        # Train
        model.learn_one(x_dict, y_true)
        
        # --- ETAP B: Buforowanie dla detektora SDDM ---
        X_batch.append([x_dict["cecha_0"], x_dict["cecha_1"], x_dict["cecha_2"]])
        y_batch.append(y_true)
        
        # Kiedy uzbieramy cały batch (200 instancji), sprawdzamy dryf
        if len(X_batch) == batch_size:
            batch_count += 1
            X_arr = np.array(X_batch)
            y_arr = np.array(y_batch)
            
            # Ewaluacja batcha w SDDM
            res = sddm.process_batch(X_arr, y_arr)
            
            # Analiza wyniku
            if res["drift"]:
                # Szukamy, która cecha spowodowała dryf (największy skok Posterior)
                feature_drifts = res["posterior"]
                drifting_feature_idx = np.argmax(feature_drifts)
                drifting_feature_name = feature_names[drifting_feature_idx]
                
                print("-" * 50)
                print(f"🔴 [Instancja {i+1} | Batch {batch_count}] WYKRYTO DRYF!")
                print(f"   -> Siła dryfu (magnitude): {res['magnitude']:.4f}")
                print(f"   -> Główne źródło dryfu: {drifting_feature_name}")
                print(f"   -> Wartości odległości per cecha (0, 1, 2): {np.round(feature_drifts, 4)}")
                print(f"   -> Dokładność modelu w momencie dryfu: {metric.get():.4%}")
                print("   -> 🔄 Reakcja: Wymuszony reset klasyfikatora...")
                print("-" * 50)
                
                # Zresetowanie klasyfikatora, aby zaadaptował się do nowych warunków
                model = tree.HoeffdingTreeClassifier()
                
            else:
                # Opcjonalny log potwierdzający działanie co kilka batchy
                if batch_count % 5 == 0:
                    print(f"🟢 [Instancja {i+1} | Batch {batch_count}] Stabilnie. Dokładność: {metric.get():.4%}")
                    
            # Wyczyszczenie buforów dla kolejnego batcha
            X_batch = []
            y_batch = []

    print(f"\nKoniec przetwarzania. Końcowa dokładność: {metric.get():.4%}")

run_sddm_test()

Rozpoczęcie strumieniowania danych...

🟢 [Instancja 1000 | Batch 5] Stabilnie. Dokładność: 97.8979%
🟢 [Instancja 2000 | Batch 10] Stabilnie. Dokładność: 98.8994%
--------------------------------------------------
🔴 [Instancja 2200 | Batch 11] WYKRYTO DRYF!
   -> Siła dryfu (magnitude): 0.7345
   -> Główne źródło dryfu: cecha_1
   -> Wartości odległości per cecha (0, 1, 2): [0.702  0.7345 0.0223]
   -> Dokładność modelu w momencie dryfu: 94.6339%
   -> 🔄 Reakcja: Wymuszony reset klasyfikatora...
--------------------------------------------------
🟢 [Instancja 3000 | Batch 15] Stabilnie. Dokładność: 95.4970%
🟢 [Instancja 4000 | Batch 20] Stabilnie. Dokładność: 96.5983%

Koniec przetwarzania. Końcowa dokładność: 96.5983%
